# 03.03 双缓冲 VectorAdd

## 小节概述

本节在相同 Shape、相同 Tile 切分下编译单缓冲和双缓冲两个 VectorAdd。你将逐行观察双缓冲的预装、预取与排空，验证两个版本都覆盖相同的真实 Tile，并比较确定性的资源指标和仅供观察的耗时指标。

<strong>前置要求：</strong> 已完成 [03.02 TQue 队列基础](03.02_queue_basics.ipynb)。<strong>建议用时：</strong> 50 分钟。<strong>本节产出：</strong> 单/双缓冲指标对照和一份队列资源计算练习。


## 教程内容

### 1. 建立演示工作副本

参考工程源码较长，因此保存在 <code>src/demo</code>。下面把它复制到当前系统用户专属且可复用的临时工作目录，既避免多用户权限冲突，也不会在仓库中产生构建文件。


In [ ]:
from pathlib import Path
import getpass
import os
import re
import shutil
import statistics
import subprocess
import sys
import tempfile

previous_repo = globals().get('REPO_ROOT')
try:
    current = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    current = cached_repo.resolve()
search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([current, *current.parents])
REPO_ROOT = next(
    (p for p in search_roots if (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库内打开本 Notebook')
os.chdir(REPO_ROOT)
CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/03_double_buffer_pipeline'
SOURCE_DEMO = CHAPTER / 'src/demo'
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
WORK_ROOT = USER_TEMP_ROOT / '03_double_buffer_pipeline'
WORK = WORK_ROOT / 'demo'
PRACTICE_FILE = WORK_ROOT / 'pipeline_math_practice.py'
ANSWER = CHAPTER / 'answer/03.03_double_buffer_vector_add/answers.md'
ANSWER_CODE = CHAPTER / 'answer/03.03_double_buffer_vector_add/pipeline_math_practice.py'
if WORK.exists():
    shutil.rmtree(WORK)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE_DEMO, WORK)
print('demo workspace:', WORK)
subprocess.run(['ls', '-R', str(WORK)], check=True)


演示工程包含两个关键文件：

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>文件</th><th style='text-align: left;'>作用</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>CMakeLists.txt</code></td><td style='text-align: left;'>从同一源码构建 <code>buffer_num=1</code> 和 <code>2</code> 两个可执行程序</td></tr>
    <tr><td style='text-align: left;'><code>vector_add_pipeline.asc</code></td><td style='text-align: left;'>包含单/双缓冲 Kernel、Host 参数校验、ACL 调用、全量精度检查和指标输出</td></tr>
  </tbody>
</table>

运行下一单元，用 <code>cat</code> 查看完整 CMake 配置，并展示 Kernel 中的两段调度。


In [ ]:
subprocess.run(['cat', str(WORK / 'CMakeLists.txt')], check=True)

source_path = WORK / 'vector_add_pipeline.asc'
lines = source_path.read_text(encoding='utf-8').splitlines()
start_line = next(i for i, line in enumerate(lines) if 'ProcessSequential()' in line and 'void' in line)
end_line = next(i for i in range(start_line + 1, len(lines)) if 'void CopyIn' in lines[i])
print('\n--- ProcessSequential / ProcessPrefetch ---')
for index in range(start_line, end_line):
    print(f'{index + 1:>3}: {lines[index]}')


### 2. 为什么先 DeQue，再预取

![单缓冲与双缓冲阶段时序](./images/double_buffer_timeline.svg)

单缓冲每轮严格完成 <code>CopyIn(tile) → Compute(tile) → CopyOut(tile)</code>。双缓冲先预装 Tile 0；每轮先 DeQue 当前输入，使 depth=1 的队列变为空，然后用第二个物理槽位预取下一 Tile；从第二轮开始同时发起上一结果的写回，最后排空末尾输出。

核心调度可概括为：

<pre><code>CopyIn(0)
for tile in [0, tileCount):
    DeQue current
    if next exists: CopyIn(tile + 1)
    if previous exists: CopyOut(tile - 1)
    Compute current
CopyOut(tileCount - 1)</code></pre>

这个顺序让每个队列任一时刻至多保留一个已入队 Tensor，所以 queue depth 仍为 1。两个物理槽位分别承载当前被 Vector 使用的数据与下一块 MTE2 预取数据。


### 3. 边界与不变量

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>场景</th><th style='text-align: left;'>必须成立</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>tileCount=1</code></td><td style='text-align: left;'>不预取、不执行 <code>tile-1</code>，计算后由最终排空写回 Tile 0</td></tr>
    <tr><td style='text-align: left;'><code>tileCount=2</code></td><td style='text-align: left;'>第 0 轮预取 Tile 1，第 1 轮写回 Tile 0，循环后写回 Tile 1</td></tr>
    <tr><td style='text-align: left;'>任意正整数 Tile 数</td><td style='text-align: left;'>CopyIn、Compute、CopyOut 对每个 Tile 各执行一次且顺序一致</td></tr>
    <tr><td style='text-align: left;'>单缓冲与双缓冲</td><td style='text-align: left;'>相同 <code>tileCount</code>、<code>tileLength</code> 和 GM 覆盖；只有调度和队列有效载荷变化</td></tr>
  </tbody>
</table>


### 4. 编译两个版本

下面在干净构建目录中生成两个可执行文件。若 CMake 找不到 ASC，请确认使用 CANN Notebook 内核，并检查 CANN 环境是否已加载。


In [ ]:
BUILD = WORK / 'build'
configure = subprocess.run(
    ['cmake', '-S', str(WORK), '-B', str(BUILD)],
    text=True, capture_output=True, check=False,
)
print(configure.stdout)
print(configure.stderr)
if configure.returncode != 0:
    raise RuntimeError('CMake 配置失败，请从上方第一条错误开始排查')
build = subprocess.run(
    ['cmake', '--build', str(BUILD), '-j'],
    text=True, capture_output=True, check=False,
)
print(build.stdout)
print(build.stderr)
if build.returncode != 0:
    raise RuntimeError('编译失败，请从上方第一条错误开始排查')

def find_executable(name):
    matches = [p for p in BUILD.rglob(name) if p.is_file()]
    if not matches:
        matches = [p for p in BUILD.rglob(name + '.exe') if p.is_file()]
    if not matches:
        raise FileNotFoundError(name)
    return matches[0]

SINGLE_EXE = find_executable('vector_add_single_buffer')
DOUBLE_EXE = find_executable('vector_add_double_buffer')
print('single:', SINGLE_EXE)
print('double:', DOUBLE_EXE)


### 5. 在相同数据切分下运行

使用 <code>N=1048576</code>、<code>blockDim=8</code>、<code>tileCount=64</code>。每个 Block 仍处理 64 个真实 Tile；双缓冲不会把循环次数减半。程序对全部输出与 CPU Golden 比较，并报告每核队列有效载荷。

每个版本运行三次，Notebook 记录内部平均耗时的中位数。耗时只用于观察，不设置双缓冲必须快多少的断言。


In [ ]:
METRIC_RE = re.compile(r'([a-z_]+)=([^\s]+)')
common_args = [
    '--length', '1048576', '--block-dim', '8', '--tile-count', '64',
    '--warmup', '10', '--iterations', '50', '--seed', '19',
]

def run_variant(executable, repeats=3):
    metrics = []
    for run_index in range(repeats):
        result = subprocess.run(
            [str(executable), *common_args],
            text=True, capture_output=True, check=False, timeout=180,
        )
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f'{executable.name} 运行失败，退出码 {result.returncode}')
        lines = [line for line in result.stdout.splitlines() if line.startswith('METRIC ')]
        if not lines:
            raise RuntimeError('未找到 METRIC 输出')
        metrics.append(dict(METRIC_RE.findall(lines[-1])))
    return metrics

single_runs = run_variant(SINGLE_EXE)
double_runs = run_variant(DOUBLE_EXE)


In [ ]:
single = single_runs[-1]
double = double_runs[-1]
deterministic_fields = ['length', 'block_dim', 'tile_count', 'tile_length', 'tile_bytes']
for field in deterministic_fields:
    assert single[field] == double[field], (field, single[field], double[field])
assert single['buffer_num'] == '1' and double['buffer_num'] == '2'
assert single['queue_depth'] == double['queue_depth'] == '1'
assert single['schedule'] == 'sequential' and double['schedule'] == 'prefetch'
assert single['correctness'] == double['correctness'] == 'PASS'
assert int(double['queue_bytes']) == 2 * int(single['queue_bytes'])
assert single['tile_count'] == double['tile_count'] == '64'

single_us = statistics.median(float(item['avg_kernel_us']) for item in single_runs)
double_us = statistics.median(float(item['avg_kernel_us']) for item in double_runs)
print('single median avg_kernel_us:', single_us)
print('double median avg_kernel_us:', double_us)
print('observed speedup (not a pass gate):', single_us / double_us)
print('queue bytes:', single['queue_bytes'], '->', double['queue_bytes'])


<strong>确定性判据：</strong> 两个版本都应 <code>correctness=PASS</code>，<code>queue_depth=1</code>，并具有相同的 <code>tile_count/tile_length</code>；双缓冲的 <code>queue_bytes</code> 是单缓冲两倍。默认小规模配置的对应值为 3072 与 6144 Byte，本次大规模配置的数值由程序按相同公式报告。

<strong>计时口径：</strong> <code>avg_kernel_us</code> 是多次 Kernel launch 加一次 stream synchronize 后取平均，包含摊薄后的 Host 启动与同步开销。运行噪声、频率状态和 Tile 大小都可能影响观察结果，因此本节不把加速比作为正确性门槛；设备侧 Task Duration 的严格分析留给实验 8。


## 课后代码实践

独立补全下面的资源计算函数。输入是正整数 <code>tile_length</code> 和 <code>buffer_num∈{1,2}</code>，输出是两个输入队列和一个输出队列的每核有效载荷字节数。不要把 <code>tileCount</code> 或 <code>blockDim</code> 乘进结果。

初始化单元已准备当前用户的临时实践文件，下一单元通过变量绝对路径写入，不会切换 Notebook 工作目录。请在 TODO 处完成公式后，运行紧随其后的自检单元并确认默认配置得到 3072/6144。


In [ ]:
%%writefile {PRACTICE_FILE}
def queue_payload_bytes(tile_length, buffer_num, tensor_count=3, dtype_bytes=4):
    if tile_length <= 0 or buffer_num not in (1, 2):
        raise ValueError('invalid pipeline configuration')
    # TODO：返回三个队列的每核有效载荷字节数。
    return 0


single = queue_payload_bytes(256, 1)
double = queue_payload_bytes(256, 2)
print('single:', single)
print('double:', double)
assert single == 3072
assert double == 6144
assert double == 2 * single


In [ ]:
practice_file = PRACTICE_FILE
practice_result = subprocess.run(
    [sys.executable, str(practice_file)],
    text=True, capture_output=True, check=False,
)
print(practice_result.stdout)
if practice_result.stderr:
    print(practice_result.stderr)
if practice_result.returncode == 0:
    print('PRACTICE PASS')
else:
    print('PRACTICE TODO：starter 失败是预期现象；补全上一个单元的 TODO 后重新运行。')


starter 的断言失败会被上一个自检单元捕获并显示为 <code>PRACTICE TODO</code>，不会中断 Notebook。补全 TODO 后重新运行写入和自检两个单元，应显示 <code>PRACTICE PASS</code>。

### 独立完成后查看参考答案


In [ ]:
SHOW_ANSWER = False
if SHOW_ANSWER:
    subprocess.run(['cat', str(ANSWER)], check=True)
    print('\n--- 代码参考实现 ---')
    subprocess.run(['cat', str(ANSWER_CODE)], check=True)
else:
    print('通过自己的资源计算后，将 SHOW_ANSWER 改为 True。')


## 本节小结

双缓冲的关键不是把常量 1 改成 2，而是让第二个物理槽位出现在可观察的预取窗口中，并正确完成 prologue 与 epilogue。进入 03.04 章节实践后，你将只修改一个小型调度头文件，用 Host 模拟先证明调度，再用 NPU 验证完整数值结果。


完成后继续进入 [03.04 章节实践](03.04_chapter_test.ipynb)。
